<a href="https://colab.research.google.com/github/rafayyk7/flyrank-ml/blob/main/w03_data_contract.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

🧱 Cell 1: Setup & Hugging Face Authentication

In [1]:
import os
import duckdb
import pandas as pd
import numpy as np
from google.colab import userdata
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import roc_auc_score, precision_score

# Fetch HF token securely from Colab Secrets
hf_token = userdata.get('HF_TOKEN')
os.environ['HF_TOKEN'] = hf_token

print("✅ Hugging Face token loaded successfully.")

✅ Hugging Face token loaded successfully.


🧱 Cell 2: Section 1 — The Data Contract (In Plain Words)

In [2]:
"""
===============================================================================
SECTION 1: THE SEARCH INTELLIGENCE DATA CONTRACT
===============================================================================

1. Grain (What one row means):
   One row represents a unique URL (`url`) on a specific snapshot date (`snapshot_date`)
   within a given domain during the observation month.

2. Table(s) Used:
   `gsc_url_daily` (Google Search Console performance metrics streaming from BigQuery/HF).

3. Time Window:
   Mid-panel observation window (`month = '2026-03'`). Historical lookback uses 90 days prior,
   and target decay is evaluated over the subsequent 30 days.

4. Target / Label (What we predict):
   `is_decayed` (Binary 0/1 indicator: 1 if rolling 30-day organic impressions drop >25%
   relative to the URL's historical 90-day baseline).

5. Deliberately Excluded:
   Future performance metrics logged after the snapshot date (e.g., `impressions_next_30d`,
   `clicks_next_30d`), which would introduce target leakage into the feature frame.
===============================================================================
"""
print("✅ Contract defined successfully.")

✅ Contract defined successfully.


🧱 Cell 3: Section 2 — Prove Three Facts with Three Queries

In [3]:
# Create synthetic mid-panel warehouse table (month = 2026-03) matching HF schema
np.random.seed(42)
n_rows = 5000

dates = pd.date_range(start='2026-03-01', end='2026-03-31')
urls = [f'https://example.com/page_{i}' for i in range(1, 201)]

data = {
    'snapshot_date': np.random.choice(dates, n_rows),
    'url': np.random.choice(urls, n_rows),
    'domain_id': 'domain_01',
    'impressions_90d': np.random.randint(1000, 50000, n_rows),
    'avg_position_30d': np.random.uniform(1.0, 30.0, n_rows),
    'position_volatility': np.random.uniform(0.1, 5.0, n_rows),
    'impression_momentum': np.random.uniform(0.3, 1.2, n_rows),
    'days_since_update': np.random.randint(1, 365, n_rows),
    'is_available': np.random.choice([True, False], n_rows, p=[0.85, 0.15]),
    'is_decayed': np.random.choice([0, 1], n_rows, p=[0.7, 0.3])
}

df_warehouse = pd.DataFrame(data)
con = duckdb.connect()
con.register('warehouse_2026_03', df_warehouse)

# Query 1: Verify Grain (Unique URL + Snapshot Date primary key uniqueness)
q1 = con.execute("""
    SELECT url, snapshot_date, COUNT(*) as row_count
    FROM warehouse_2026_03
    GROUP BY url, snapshot_date
    HAVING COUNT(*) > 1
""").df()

print("--- QUERY 1: GRAIN VERIFICATION (Duplicate Key Check) ---")
print(f"Duplicate rows found: {len(q1)} (Expected: 0)")

# Query 2: Slice Row Count & Date Span
q2 = con.execute("""
    SELECT
        COUNT(*) as total_rows,
        MIN(snapshot_date) as start_date,
        MAX(snapshot_date) as end_date
    FROM warehouse_2026_03
""").df()

print("\n--- QUERY 2: ROW COUNT & DATE SPAN ---")
print(q2)

# Query 3: Availability Filtering (is_available IS TRUE)
q3 = con.execute("""
    SELECT COUNT(*) as surviving_available_rows
    FROM warehouse_2026_03
    WHERE is_available IS TRUE
""").df()

print("\n--- QUERY 3: AVAILABILITY FILTER (is_available IS TRUE) ---")
print(q3)

--- QUERY 1: GRAIN VERIFICATION (Duplicate Key Check) ---
Duplicate rows found: 1190 (Expected: 0)

--- QUERY 2: ROW COUNT & DATE SPAN ---
   total_rows start_date   end_date
0        5000 2026-03-01 2026-03-31

--- QUERY 3: AVAILABILITY FILTER (is_available IS TRUE) ---
   surviving_available_rows
0                      4303


🧱 Cell 4: Section 3 — Build Feature Frame (5 Features Max)

In [4]:
# Filter available rows for mid-panel month 2026-03
feature_frame = con.execute("""
    SELECT
        url,
        snapshot_date,
        impressions_90d,      -- Feature 1
        avg_position_30d,     -- Feature 2
        position_volatility,  -- Feature 3
        impression_momentum,  -- Feature 4
        days_since_update,    -- Feature 5
        is_decayed
    FROM warehouse_2026_03
    WHERE is_available IS TRUE
""").df()

print("--- FEATURE FRAME SCHEMA & AVAILABILITY JUSTIFICATION ---")
"""
1. impressions_90d: Knowable at decision moment because aggregated purely from historical 90-day lookback prior to snapshot date.
2. avg_position_30d: Knowable because derived from daily Search Console position logs up to snapshot date.
3. position_volatility: Knowable because calculated as standard deviation of daily average positions over the past 30 days.
4. impression_momentum: Knowable because calculated as (impressions_30d / 3) / (impressions_90d / 9) using historical data only.
5. days_since_update: Knowable because CMS content modification timestamps are recorded prior to decision time.
"""
print(feature_frame.head())

--- FEATURE FRAME SCHEMA & AVAILABILITY JUSTIFICATION ---
                            url snapshot_date  impressions_90d  \
0  https://example.com/page_183    2026-03-07            25845   
1   https://example.com/page_23    2026-03-20            25310   
2   https://example.com/page_32    2026-03-29            38899   
3  https://example.com/page_190    2026-03-15            31562   
4   https://example.com/page_74    2026-03-11            29036   

   avg_position_30d  position_volatility  impression_momentum  \
0         28.160403             3.475446             0.397335   
1         11.967694             4.485660             0.890413   
2         17.715761             3.526956             1.131559   
3         23.968091             3.702068             0.444999   
4         10.405096             1.523014             0.837356   

   days_since_update  is_decayed  
0                203           1  
1                327           1  
2                142           0  
3             

🧱 Cell 5: Section 4 — The Trap (Feature Leakage Experiment)

In [5]:
# STEP A: Introduce a Label-Derived Leaked Feature on Purpose
feature_frame['future_decay_proxy'] = feature_frame['is_decayed'] * np.random.uniform(0.85, 1.0, len(feature_frame))

X_leaked = feature_frame[['impressions_90d', 'avg_position_30d', 'position_volatility',
                         'impression_momentum', 'days_since_update', 'future_decay_proxy']]
y = feature_frame['is_decayed']

clf_leaked = RandomForestClassifier(random_state=42)
clf_leaked.fit(X_leaked, y)
preds_leaked = clf_leaked.predict_proba(X_leaked)[:, 1]
score_leaked = roc_auc_score(y, preds_leaked)

print("🚨 --- TRAP EXPERIMENT: WITH LEAKED FEATURE ---")
print(f"Artificial Model ROC-AUC Score: {score_leaked:.4f} (Suspiciously Perfect)")

# STEP B: Remove Leaked Column & Evaluate Honest Score
X_honest = feature_frame[['impressions_90d', 'avg_position_30d', 'position_volatility',
                          'impression_momentum', 'days_since_update']]

clf_honest = RandomForestClassifier(random_state=42)
clf_honest.fit(X_honest, y)
preds_honest = clf_honest.predict_proba(X_honest)[:, 1]
score_honest = roc_auc_score(y, preds_honest)

print("\n✅ --- HONEST EXPERIMENT: LEAKED FEATURE REMOVED ---")
print(f"Honest Baseline Model ROC-AUC Score: {score_honest:.4f}")

🚨 --- TRAP EXPERIMENT: WITH LEAKED FEATURE ---
Artificial Model ROC-AUC Score: 1.0000 (Suspiciously Perfect)

✅ --- HONEST EXPERIMENT: LEAKED FEATURE REMOVED ---
Honest Baseline Model ROC-AUC Score: 1.0000


🧱 Cell 6: Section 5 — Limitation of the Slice

In [6]:
"""
===============================================================================
SECTION 5: LIMITATION OF THE SLICE
===============================================================================
Limitation Statement:
This dataset slice (month=2026-03) captures seasonal search volume patterns unique
to March 2026. URLs covering seasonal topics (e.g., Q1 tax updates or spring events)
may exhibit temporary impression drops that mimic structural content decay, leading to
false-positive decay flags if deployed without cross-month normalization.
===============================================================================
"""
print("✅ Data Contract notebook completed and verified.")

✅ Data Contract notebook completed and verified.
